In [7]:
import pandas as pd
import numpy as np

In [8]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\Admin\Desktop\Customer_Churn_Prediction\Customer_Churn_Prediction 2
['churn_features_models.ipynb']


In [9]:
os.listdir("../Customer_Churn_Prediction")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../Customer_Churn_Prediction'

In [10]:
os.listdir(r"C:\Users\Admin\Desktop")

['.vscode',
 'Bank_ML_Project',
 'brindham.session.sql',
 'Customer_Churn_Prediction',
 'desktop.ini',
 'Microsoft Edge.lnk',
 'Postman.lnk',
 'python',
 'Retail_Order_API',
 'SQL',
 'SQL_task',
 'task',
 'Visual Studio Code.lnk']

In [11]:
os.listdir(r"C:\Users\Admin\Desktop\Customer_Churn_Prediction")

['.git',
 'churn_eda.ipynb',
 'Customer_Churn_Prediction 2',
 'data',
 'data.py',
 'ml.py',
 'models.py',
 'output.py',
 'README.md']

In [12]:
os.listdir(r"C:\Users\Admin\Desktop\Customer_Churn_Prediction\data")

['WA_Fn-UseC_-Telco-Customer-Churn.csv']

In [13]:
df = pd.read_csv(
    r"C:\Users\Admin\Desktop\Customer_Churn_Prediction\data\WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [14]:
df.shape

(7043, 21)

In [15]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [17]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [18]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["New", "Regular", "Loyal", "Long-term"]
)

In [19]:
df["spend_per_month"] = df["MonthlyCharges"]

In [20]:
service_columns = [
    "PhoneService",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["service_count"] = (df[service_columns] != "No").sum(axis=1)

In [21]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df["average_monthly_spend"] = (
    df["TotalCharges"] / df["tenure"].replace(0, np.nan)
)

In [22]:
df[[
    "tenure_group",
    "spend_per_month",
    "service_count",
    "average_monthly_spend"
]].head()

,tenure_group,spend_per_month,service_count,average_monthly_spend
0,New,29.85,2,29.850000
1,Loyal,56.95,4,55.573529
2,New,53.85,4,54.075000
3,Loyal,42.30,4,40.905556
4,New,70.70,2,75.825000


In [23]:
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes": 1, "No": 0})

In [25]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'spend_per_month', 'service_count', 'average_monthly_spend']
Categorical features: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']


C:\Users\Admin\AppData\Local\Temp\ipykernel_10348\499034265.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


In [26]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

NameError: name 'Pipeline' is not defined

In [27]:
from sklearn.pipeline import Pipeline

In [28]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [29]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [30]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [31]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [32]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

In [33]:
from sklearn.dummy import DummyClassifier

dummy_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

In [34]:
from sklearn.model_selection import cross_validate

scoring = {
    "roc_auc": "roc_auc",
    "f1": "f1"
}

In [35]:
lr_scores = cross_validate(
    lr_pipeline,
    X,
    y,
    cv=5,
    scoring=scoring
)

In [36]:
rf_scores = cross_validate(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring=scoring
)

In [37]:
dummy_scores = cross_validate(
    dummy_pipeline,
    X,
    y,
    cv=5,
    scoring=scoring
)

In [38]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Dummy Classifier"
    ],
    "CV ROC-AUC": [
        lr_scores["test_roc_auc"].mean(),
        rf_scores["test_roc_auc"].mean(),
        dummy_scores["test_roc_auc"].mean()
    ],
    "CV F1": [
        lr_scores["test_f1"].mean(),
        rf_scores["test_f1"].mean(),
        dummy_scores["test_f1"].mean()
    ]
})

results

,Model,CV ROC-AUC,CV F1
0,Logistic Regression,0.845703,0.593989
1,Random Forest,0.830219,0.547010
2,Dummy Classifier,0.500000,0.000000


In [39]:
results.to_csv("baseline_results.csv", index=False)